In [ ]:
# idk havent found a better way sorry
import os, sys
sys.path.insert(0, os.path.abspath("../.."))

import numpy as np
import matplotlib.pyplot as plt
from src import fdm_schemes
from src.iter_schemes import jacobi, gauss_seidel, sor
from src.grid import rectangle_sink, combine_sinks, make_grid, empty_insulator, rectangle_insulator

## 1.6 

### H

In [ ]:
N= 50

# cJ, dJ = jacobi(N)
# cG, dG = gauss_seidel(N)
cS, dS, _, _ = sor(N, omega=1.8)   





In [ ]:
plt.imshow(cJ, aspect='auto')
plt.colorbar()
plt.show()


In [ ]:
y = np.linspace(1, 0, N)      
analytic = y                     

profJ = cJ.mean(axis=1)
profG = cG.mean(axis=1)
profS = cS.mean(axis=1)

errJ = np.max(np.abs(profJ - analytic))
errG = np.max(np.abs(profG - analytic))
errS = np.max(np.abs(profS - analytic))

print("Max abs error vs c(y)=y:")
print("Jacobi      :", errJ)
print("Gauss-Seidel:", errG)
print("SOR (1.8)   :", errS)


In [ ]:
plt.figure()
plt.plot(y, analytic, label="Analytical solution")
plt.plot(y, profJ, "--", label="Jacobi")
plt.plot(y, profG, "--", label="Gauss-Seidel")
plt.plot(y, profS, "--", label="SOR ω=1.8")
plt.xlabel("y")
plt.ylabel("c")
plt.legend()
plt.title("Comparison of Iterative Schemes (mean of x) for c(y)=y")
plt.show()


In [ ]:

def calculate_and_plot_mse(c, N, analytic):

    mse_errors = []
    
    for i in range(N):
        mse = np.mean((c[i, :] - analytic[i]) ** 2)
        mse_errors.append(mse)
    max_mse = max(mse_errors)
    return mse_errors, max_mse


mseJ, maxJ = calculate_and_plot_mse(cJ, N, analytic)
mseG, maxG = calculate_and_plot_mse(cG, N, analytic)
mseS, maxS = calculate_and_plot_mse(cS, N, analytic)

plt.figure(figsize=(10, 6))
plt.plot(mseJ, marker='o', label="Jacobi")
plt.plot(mseG, marker='o', label="Gauss-Seidel")
plt.plot(mseS, marker='o', label="SOR, ω=1.8")
plt.text(4, 0.9*maxJ, f'Max MSE Jacobi: {maxJ:.2e}', fontsize=12, color='tab:blue')
plt.text(17, 2.9*10**(-5), f'Max MSE Gauss-Seidel: {maxG:.2e}', fontsize=12, color='tab:orange')
plt.text(18, 0.8*10**(-5), f'Max MSE SOR: {maxS:.2e}', fontsize=12, color='tab:green')

plt.xlabel("x", fontsize=12)
plt.ylabel("Mean Squared Error", fontsize=12)
plt.title("Point-wise MSE", fontsize=15.5)
plt.grid()
plt.legend()
plt.show()

### I

In [ ]:
_, dj_2 = jacobi(N)
_, dg_2 = gauss_seidel(N)

omegas = [1.2, 1.5, 1.8, 1.9]
ds_2 = {}
for o in omegas:
    _, ds_2[o], _ = sor(N, omega=o)


In [ ]:
plt.figure(figsize=(7,5))

plt.semilogy(dj_2, label="Jacobi", alpha=0.9)
plt.semilogy(dg_2, label="Gauss–Seidel", alpha=0.9)

for o in omegas:
    plt.semilogy(ds_2[o], "--", label=f"SOR ω={o}", alpha=0.5)


plt.xlabel("k")
plt.ylabel(r"$\delta(k)$")
plt.title("Convergence Comparison for Iterative Schemes")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()


### J

In [ ]:
N=50
omegas = np.arange(0.7, 2.3, 0.05) 

ks = []
for o in omegas:
    _, _, k, conv = sor(N, omega=o)
    ks.append(k if conv else np.inf)

ks = np.array(ks)

w0 = omegas[np.argmin(ks)]
print(f"Initial best: N={N}, ω={w0:.2f}, k={ks.min()}") # based on min k, cos what else but open to suggestoiins lol

plt.figure()
plt.plot(omegas, ks, marker="o")
plt.xlabel("ω")
plt.ylabel("k to convergence")
plt.title(f"SOR Convergence vs ω (N={N})")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# based on initial best, more precise check cos looks like maybe a smalller min somehere there??

w0 = 1.90
omegas_more = np.arange(w0 - 0.08, w0 + 0.081, 0.01) # why 0.08 idk but eh

ks_2 = []
for o in omegas_more:
    _, _, k, conv = sor(N, o)
    ks_2.append(k if conv else np.inf)

ks_2 = np.array(ks_2)

o_opt = omegas_more[np.argmin(ks_2)]
k_opt = ks_2.min()
print(f"Best: N={N}, ω_opt={o_opt:.2f}, k={k_opt}")

In [ ]:
plt.figure()
plt.plot(omegas_more, ks_2, marker="o")
plt.xlabel("ω")
plt.ylabel(" k to convergence")
plt.title(f"Refined SOR Convergence (N={N})")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# more properly
Ns = [20, 30, 50, 80, 100] # prob do more later when time
tol = 1e-5
max_iter = 50000

o_opts = []
k_opts = []

for N in Ns:
    omegas = np.arange(1.0, 2.0, 0.05)
    ks = []
    for o in omegas:
        _, _, k, conv = sor(N, omega=o, tol=tol, max_iter=max_iter)
        ks.append(k if conv else np.inf)
    ks = np.array(ks)

    w0 = omegas[np.argmin(ks)]

    omegas_finer = np.arange(max(1.0, w0-0.08), min(1.99, w0+0.08)+1e-12, 0.01)
    ks_finer = []
    for o in omegas_finer:
        _, _, k, conv = sor(N, omega=o, tol=tol, max_iter=max_iter)
        ks_finer.append(k if conv else np.inf)
    ks_finer = np.array(ks_finer)

    o_opt = omegas_finer[np.argmin(ks_finer)]
    k_opt = ks_finer.min()

    o_opts.append(o_opt)
    k_opts.append(k_opt)

   



In [ ]:
plt.figure()
plt.plot(Ns, o_opts, marker="o")
plt.xlabel("Grid size N")
plt.ylabel("Optimal ω")
plt.title("Optimal SOR relaxation parameter vs grid size")
plt.grid(True, alpha=0.3)
plt.show()

## K)

## placement impact

In [ ]:
def sink_area(sink):
    return int(np.sum(sink))


In [ ]:
def square_sink(N, i0, j0, L):
    return rectangle_sink(N, i0, i0+L, j0, j0+L)


In [ ]:
N = 50
w = 1.9
tol = 1e-5

# same shape: 10x6 = 60 cells
sink_top = rectangle_sink(N, 1, 7, 2,  10)   
sink_mid = rectangle_sink(N, 22, 28, 0, 30)
sink_bot = rectangle_sink(N, 0, 0, 0, 0)   

# for name, s in [("top", sink_top), ("mid", sink_mid), ("bottom", sink_bot)]:
#     c, d, k = sor(N, omega=w, tol=tol, sink=s)
#     print(name, "area", sink_area(s), "iters", k)


c_s, d, k = sor(N, omega=w, tol=tol, sink=sink_top)
print(k)

In [ ]:


def placement_sweep(N, omega, L=8, step=5, tol=1e-5, max_iter=10000):
    """
    Sweep an LxL sink across interior placements in steps of 'step'.
    Returns:
      i_starts, j_starts: arrays of start indices used
      K: 2D array with iterations (shape len(i_starts) x len(j_starts))
    """
    # valid i range: 1..N-2-L+1 so the square stays off top/bottom boundaries
    i_starts = np.arange(1, (N-1) - L + 1, step)
    j_starts = np.arange(0, N - L + 1, step)  # keep inside range (no wrap)

    K = np.empty((len(i_starts), len(j_starts)), dtype=float)

    for a, i0 in enumerate(i_starts):
        for b, j0 in enumerate(j_starts):
            s = square_sink(N, i0, j0, L)
            _, _, k = sor(N, omega=omega, tol=tol, max_iter=max_iter, sink=s)
            K[a, b] = k if k < max_iter else np.inf

    return i_starts, j_starts, K

In [ ]:
N = 50
tol = 1e-5
max_iter = 10000
omega = 1   # or use ω_opt from J (baseline)

i_starts, j_starts, K = placement_sweep(N, omega, L=6, step=3, tol=tol, max_iter=max_iter)

print("min k:", np.nanmin(K), "max k:", np.nanmax(K))

In [ ]:
plt.figure()
plt.imshow(K, origin="lower", aspect="auto")
plt.colorbar(label="iterations k")
plt.xticks(range(len(j_starts)), j_starts)
plt.yticks(range(len(i_starts)), i_starts)
plt.xlabel("j0 (square left edge)")
plt.ylabel("i0 (square top edge)")
plt.title(f"Iteration count vs sink placement (L=6, step=3, ω={omega})")
plt.show()

## area checks

In [ ]:
def random_square_sink(N, L, rng):
    i0 = rng.integers(1, (N-1) - L + 1)
    j0 = rng.integers(0, N - L + 1)
    return rectangle_sink(N, i0, i0+L, j0, j0+L)

In [ ]:
def area_sweep_squares(N, omega, L_values, n_trials=10, tol=1e-5, max_iter=10000, seed=0):
    rng = np.random.default_rng(seed)

    areas = []
    k_mean = []
    k_std = []
    k_ci95 = []
    k_all = {}  

    for L in L_values:
        ks = []
        for _ in range(n_trials):
            sink = random_square_sink(N, L, rng)
            _, _, k, conv = sor(N, omega=omega, tol=tol, max_iter=max_iter, sink=sink)
            ks.append(k if conv else np.inf)

        ks = np.array(ks, dtype=float)
        m = np.nanmean(ks)
        s = np.nanstd(ks, ddof=1)
        n = np.sum(~np.isnan(ks))
        ci95 = 1.96 * (s / np.sqrt(n))
        k_ci95.append(ci95)
        areas.append(L*L)
        k_mean.append(np.nanmean(ks))
        k_std.append(np.nanstd(ks))
        k_all[L] = ks

    return np.array(areas), np.array(k_mean), np.array(k_std), np.array(k_ci95), k_all

In [ ]:
N = 50
omega = 1.9
tol = 1e-5
max_iter = 10000

L_values = np.arange(1,  46, 1)  # to avoid errors from  i0 = rng.integers(1, (N-1) - L + 1)
areas, mean_k, std_k, ci95_k, k_all = area_sweep_squares(
    N, omega, L_values, n_trials=50, tol=tol, max_iter=max_iter, seed=1
)



In [ ]:
plt.figure()
plt.errorbar(areas, mean_k, yerr=ci95_k, fmt="o-")
plt.xlabel("Square sink area ")
plt.ylabel("Iterations to converge k (mean ± 95% CI)")
plt.title(f"Effect of sink area on convergence (N={N}, ω={omega}), trials = 50")
plt.grid(True, alpha=0.3)
plt.show()

## finding optimal omega

In [ ]:
def omega_opt_for_case(N, sink=None, tol=1e-5, max_iter=10000,
                       omega_min=1.0, omega_max=1.95, domega=0.05,
                       refine_halfwidth=0.08, refine_step=0.01):

    omegas = np.arange(omega_min, omega_max + 1e-12, domega)
    ks = []

    for w in omegas:
        _, _, k, conv = sor(N, omega=w, tol=tol, max_iter=max_iter, sink=sink)
        ks.append(k if conv else np.inf)

    ks = np.array(ks)
    w0 = omegas[np.argmin(ks)]

    omegas_f = np.arange(max(omega_min, w0-refine_halfwidth),
                         min(1.99, w0+refine_halfwidth) + 1e-12,
                         refine_step)

    ks_f = []
    for w in omegas_f:
        _, _, k, conv = sor(N, omega=w, tol=tol, max_iter=max_iter, sink=sink)
        ks_f.append(k if conv else np.inf)

    ks_f = np.array(ks_f)
    w_opt = omegas_f[np.argmin(ks_f)]
    k_opt = ks_f.min()

    return w_opt, k_opt, (omegas, ks), (omegas_f, ks_f)

In [ ]:
N = 50

# medium square in center
sink_sq = rectangle_sink(N, 20, 30, 20, 30)  # 10x10

# same area (100) but higher perimeter: 5x20 bar
sink_bar = rectangle_sink(N, 22, 27, 15, 35)  # 5x20

# same square near top and near bottom
sink_sq_top = rectangle_sink(N, 2, 12, 20, 30)
sink_sq_bot = rectangle_sink(N, 37, 47, 20, 30)  # keep < N-1

# two objects, total area 100 (two 5x10 rectangles)
s1 = rectangle_sink(N, 10, 15, 5, 15)   # 5x10
s2 = rectangle_sink(N, 30, 35, 35, 45)  # 5x10
sink_two = combine_sinks(s1, s2)

In [ ]:
tol = 1e-5
max_iter = 10000

cases = {
    "no sink": None,
    "square center (10x10)": sink_sq,
    "bar (5x20)": sink_bar,
    "square top (10x10)": sink_sq_top,
    "square bottom (10x10)": sink_sq_bot,
    "two rectangles (2x 5x10)": sink_two,
}

results = {}
for name, s in cases.items():
    w_opt, k_opt, _, _ = omega_opt_for_case(N, sink=s, tol=tol, max_iter=max_iter)
    results[name] = (w_opt, k_opt)
    print(f"{name:28s} ω_opt={w_opt:.2f}, k_min={k_opt}")

## J)